In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
mpl.rcParams['pdf.fonttype'] = 42

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import glob

# Define paths for each method
base_path = Path("/QRISdata/Q1851/Xiao/Working_project/benchmarking")
methods = {
    'virchow2_BLEEP': base_path / "BLEEP/Skin_data",
    'virchow2_DEEPPT': base_path / "DEEPPT/Skin_data",
    'virchow2_DeepSpace': base_path / "DeepSpace/Skin_data",
    'virchow2_STimage': base_path / "STimage/Skin_data"
}

# Collect all correlation data
all_correlations = []

for method_name, method_path in methods.items():
    # Find all correlation files for this method
    cor_files = glob.glob(str(method_path / "*clinical_cor.csv"))
    
    print(f"\n{method_name}: Found {len(cor_files)} files")
    
    for cor_file in cor_files:
        try:
            # Read correlation file
            df_cor = pd.read_csv(cor_file, index_col=0)
            
            print(f"  {Path(cor_file).name}")
            print(f"    Shape: {df_cor.shape}")
            print(f"    Columns: {df_cor.columns.tolist()}")
            print(f"    First few rows:")
            print(df_cor.head())
            
            # Check if 'r' column exists
            if 'r' not in df_cor.columns:
                print(f"    WARNING: 'r' column not found! Available columns: {df_cor.columns.tolist()}")
                # Try to find the correlation column
                if len(df_cor.columns) == 1:
                    df_cor.columns = ['r']
                    print(f"    Renamed single column to 'r'")
                elif 'Pearson correlation' in df_cor.columns:
                    # Handle STimage format
                    df_cor = df_cor.rename(columns={'Pearson correlation': 'r'})
                    print(f"    Renamed 'Pearson correlation' to 'r'")
            
            # Reset index to avoid duplicate index issues
            df_cor = df_cor.reset_index()
            
            # Standardize column names
            if len(df_cor.columns) == 2:
                df_cor.columns = ['gene', 'r']
            elif 'r' in df_cor.columns:
                # Keep only gene and r columns if there are extras
                if 'Gene' in df_cor.columns:
                    df_cor = df_cor[['Gene', 'r']]
                    df_cor.columns = ['gene', 'r']
                elif df_cor.columns[0] != 'gene':
                    # First column is likely the gene name
                    gene_col = df_cor.columns[0]
                    df_cor = df_cor[[gene_col, 'r']]
                    df_cor.columns = ['gene', 'r']
            
            # Add method name and file info
            df_cor['Method'] = method_name
            df_cor['File'] = Path(cor_file).stem
            
            all_correlations.append(df_cor)
            print(f"Loaded {len(df_cor)} genes")
            
        except Exception as e:
            print(f"ERROR loading {Path(cor_file).name}: {e}")

# Combine all data
df_all = pd.concat(all_correlations, axis=0, ignore_index=True)


print(f"Total rows: {len(df_all)}")
print(f"\nCorrelation summary by method:")
summary = df_all.groupby('Method')['r'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
print(summary)

# Create the boxplot
fig, ax = plt.subplots(figsize=(20, 6))

# Define colors matching your image
colors = ['#4472C4', '#ED7D31', '#70AD47', '#C55A5A']
method_order = ['virchow2_BLEEP', 'virchow2_DEEPPT', 'virchow2_DeepSpace', 'virchow2_STimage']

# Filter out methods with no data
available_methods = [m for m in method_order if m in df_all['Method'].unique() and 
                     df_all[df_all['Method'] == m]['r'].notna().sum() > 0]
colors_filtered = [colors[method_order.index(m)] for m in available_methods]

print(f"\nAvailable methods for plotting: {available_methods}")

# Create horizontal boxplot using seaborn
sns.boxplot(data=df_all[df_all['Method'].isin(available_methods)], 
            y='Method', x='r', 
            order=available_methods,
            palette=colors_filtered,
            hue='Method',
            legend=False,
            ax=ax,
            showfliers=True,
            flierprops=dict(marker='o', markersize=3, alpha=0.5))

pretty_names = {
    'virchow2_BLEEP': 'BLEEP',
    'virchow2_DEEPPT': 'DEEPPT',
    'virchow2_DeepSpace': 'DeepSpace',
    'virchow2_STimage': 'STimage'
}

ax.set_yticklabels([pretty_names[m] for m in available_methods])

# Customize plot
ax.set_xlabel('Pearson correlation', fontsize=45, fontweight='bold')
ax.set_ylabel('Skin Dataset', fontsize=45, fontweight='bold')
ax.axvline(x=0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.tick_params(axis= 'y' , labelsize=36)
ax.tick_params(axis= 'x' , labelsize=36)

# Set x-axis limits based on data
if len(available_methods) > 0:
    # x_min = df_all[df_all['Method'].isin(available_methods)]['r'].min() - 0.05
    # x_max = df_all[df_all['Method'].isin(available_methods)]['r'].max() + 0.05
    x_min = -0.2
    x_max = 1.0
    ax.set_xlim(x_min, x_max)

plt.tight_layout()
plt.savefig('Skin_Clin.png', dpi=600, bbox_inches='tight')
plt.savefig('Skin_Clin.pdf', dpi=600, bbox_inches='tight')
plt.show()

# Print detailed statistics
print("Detailed Statistics by Method:")
for method in available_methods:
    method_data = df_all[df_all['Method'] == method]['r']
    print(f"\n{method}:")
    print(f"  Count: {len(method_data)}")
    print(f"  Mean:   {method_data.mean():.4f}")
    print(f"  Median: {method_data.median():.4f}")
    print(f"  Std:    {method_data.std():.4f}")
    print(f"  Min:    {method_data.min():.4f}")
    print(f"  Max:    {method_data.max():.4f}")
    print(f"  Q1:     {method_data.quantile(0.25):.4f}")
    print(f"  Q3:     {method_data.quantile(0.75):.4f}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import glob

# Define paths for each method
base_path = Path("/QRISdata/Q1851/Xiao/Working_project/benchmarking")
methods = {
    'virchow2_BLEEP': base_path / "BLEEP/BC_data",
    'virchow2_DEEPPT': base_path / "DEEPPT/BC_data",
    'virchow2_DeepSpace': base_path / "DeepSpace/BC_data",
    'virchow2_STimage': base_path / "STimage/BC_data"
}

# Collect all correlation data
all_correlations = []

for method_name, method_path in methods.items():
    # Find all correlation files for this method
    cor_files = glob.glob(str(method_path / "*clinical_cor.csv"))
    
    print(f"\n{method_name}: Found {len(cor_files)} files")
    
    for cor_file in cor_files:
        try:
            # Read correlation file
            df_cor = pd.read_csv(cor_file, index_col=0)
            
            print(f"  {Path(cor_file).name}")
            print(f"    Shape: {df_cor.shape}")
            print(f"    Columns: {df_cor.columns.tolist()}")
            print(f"    First few rows:")
            print(df_cor.head())
            
            # Check if 'r' column exists
            if 'r' not in df_cor.columns:
                print(f"    WARNING: 'r' column not found! Available columns: {df_cor.columns.tolist()}")
                # Try to find the correlation column
                if len(df_cor.columns) == 1:
                    df_cor.columns = ['r']
                    print(f"    Renamed single column to 'r'")
                elif 'Pearson correlation' in df_cor.columns:
                    # Handle STimage format
                    df_cor = df_cor.rename(columns={'Pearson correlation': 'r'})
                    print(f"    Renamed 'Pearson correlation' to 'r'")
            
            # Reset index to avoid duplicate index issues
            df_cor = df_cor.reset_index()
            
            # Standardize column names
            if len(df_cor.columns) == 2:
                df_cor.columns = ['gene', 'r']
            elif 'r' in df_cor.columns:
                # Keep only gene and r columns if there are extras
                if 'Gene' in df_cor.columns:
                    df_cor = df_cor[['Gene', 'r']]
                    df_cor.columns = ['gene', 'r']
                elif df_cor.columns[0] != 'gene':
                    # First column is likely the gene name
                    gene_col = df_cor.columns[0]
                    df_cor = df_cor[[gene_col, 'r']]
                    df_cor.columns = ['gene', 'r']
            
            # Add method name and file info
            df_cor['Method'] = method_name
            df_cor['File'] = Path(cor_file).stem
            
            all_correlations.append(df_cor)
            print(f"Loaded {len(df_cor)} genes")
            
        except Exception as e:
            print(f"ERROR loading {Path(cor_file).name}: {e}")

# Combine all data
df_all = pd.concat(all_correlations, axis=0, ignore_index=True)

print(f"Total rows: {len(df_all)}")
print(f"\nCorrelation summary by method:")
summary = df_all.groupby('Method')['r'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
print(summary)

# Create the boxplot
fig, ax = plt.subplots(figsize=(20, 6))

# Define colors matching your image
colors = ['#4472C4', '#ED7D31', '#70AD47', '#C55A5A']
method_order = ['virchow2_BLEEP', 'virchow2_DEEPPT', 'virchow2_DeepSpace', 'virchow2_STimage']

# Filter out methods with no data
available_methods = [m for m in method_order if m in df_all['Method'].unique() and 
                     df_all[df_all['Method'] == m]['r'].notna().sum() > 0]
colors_filtered = [colors[method_order.index(m)] for m in available_methods]

print(f"\nAvailable methods for plotting: {available_methods}")

# Create horizontal boxplot using seaborn
sns.boxplot(data=df_all[df_all['Method'].isin(available_methods)], 
            y='Method', x='r', 
            order=available_methods,
            palette=colors_filtered,
            hue='Method',
            legend=False,
            ax=ax,
            showfliers=True,
            flierprops=dict(marker='o', markersize=3, alpha=0.5))
pretty_names = {
    'virchow2_BLEEP': 'BLEEP',
    'virchow2_DEEPPT': 'DEEPPT',
    'virchow2_DeepSpace': 'DeepSpace',
    'virchow2_STimage': 'STimage'
}

ax.set_yticklabels([pretty_names[m] for m in available_methods])

# Customize plot
ax.set_xlabel('Pearson correlation', fontsize=45, fontweight='bold')
ax.set_ylabel('BC Dataset', fontsize=45, fontweight='bold')
ax.axvline(x=0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.tick_params(axis='x', labelsize=36)
ax.tick_params(axis='y', labelsize=36)

# Set x-axis limits based on data
if len(available_methods) > 0:
    # x_min = df_all[df_all['Method'].isin(available_methods)]['r'].min() - 0.05
    # x_max = df_all[df_all['Method'].isin(available_methods)]['r'].max() + 0.05
    x_min = -0.2
    x_max = 1.0
    ax.set_xlim(x_min, x_max)

plt.tight_layout()
plt.savefig('BC_Clin.png', dpi=600, bbox_inches='tight')
plt.savefig('BC_Clin.pdf', dpi=600, bbox_inches='tight')
plt.show()

# Print detailed statistics
print("Detailed Statistics by Method:")
for method in available_methods:
    method_data = df_all[df_all['Method'] == method]['r']
    print(f"\n{method}:")
    print(f"  Count: {len(method_data)}")
    print(f"  Mean:   {method_data.mean():.4f}")
    print(f"  Median: {method_data.median():.4f}")
    print(f"  Std:    {method_data.std():.4f}")
    print(f"  Min:    {method_data.min():.4f}")
    print(f"  Max:    {method_data.max():.4f}")
    print(f"  Q1:     {method_data.quantile(0.25):.4f}")
    print(f"  Q3:     {method_data.quantile(0.75):.4f}")